In [1]:
# %pip install surprise --quiet

In [2]:
import numpy as np
import pandas as pd
import re

In [3]:
data = pd.read_csv("../reviews_filtered.csv")
 # import reviews

print(data.head(10))

   Unnamed: 0  movie_id                       name  count_  user_id  rating
0           0     34222  slumber-party-massacre-ii      32   438333       7
1           1     34222  slumber-party-massacre-ii      32   437213       8
2           2     34222  slumber-party-massacre-ii      32   434710       7
3           3     34222  slumber-party-massacre-ii      32   423063       5
4           4     34222  slumber-party-massacre-ii      32   244269       9
5           5     34222  slumber-party-massacre-ii      32   188272       7
6           6     34222  slumber-party-massacre-ii      32    30514       6
7           7     34222  slumber-party-massacre-ii      32     8026      10
8           8    462065         run-sweetheart-run      32  1146451       5
9           9    462065         run-sweetheart-run      32  1067317       7


import re

love_death_robots_filter = re.compile("love-death-robots")
black_mirror_filter = re.compile("black-mirror")

to_be_removed = []

for idx, row in data.iterrows():
    if (re.match(love_death_robots_filter, row['name'])):
        to_be_removed.append(idx)
    elif (re.match(black_mirror_filter, row['name'])):
        to_be_removed.append(idx)

data = data.drop(to_be_removed)

data.to_csv("../reviews_filtered.csv")

In [4]:
unique_movies = sorted(list(set(zip(data['movie_id'], data['name']))))
unique_users = sorted(list(set(data['user_id'])))

num_movies = len(unique_movies)
num_users = len(unique_users)

print("%d Unique Movies" % num_movies)
print("%d Unique Users" % num_users)

5937 Unique Movies
2516 Unique Users


In [5]:
# these are a set of variables to make conversion between labels and attribute lookup easier
# users have two labels: their ids in the original dataset and their idxs in the list from the previous cell
# movies have 3: the same two as users, and also a name

movie_id_to_idx = {movie_id: idx for idx,
                   (movie_id, _) in enumerate(unique_movies)}
user_id_to_idx = {user_id: idx for idx, user_id in enumerate(unique_users)}

movie_name_to_idx = {movie_name: idx for idx,
                     (_, movie_name) in enumerate(unique_movies)}

movie_names = [movie_name for (_, movie_name) in unique_movies]

movie_id_to_name = {movie_id: movie_name for (
    movie_id, movie_name) in unique_movies}

# we also add a variable to easily look up how many reviews a movie has in our dataset
movie_freqs = {movie_id_to_idx[movie_id]: freq for (
    movie_id, freq) in zip(data['movie_id'], data['count_'])}


In [6]:
# loos up id and list idx for a movie given its name
# this uses fuzzy string matching so we can look up the attributes for a given movie
def look_up_movie(movie_name):
    match_string = re.compile(".*" + movie_name + ".*")
    for idx, (movie_id, movie_name) in enumerate(unique_movies):
        # movie_name = movie_id_to_name[movie_id]
        if (re.match(match_string, movie_name)):
            yield (movie_name, movie_id, idx, movie_freqs[idx])

In [7]:
# tests the above function
for (movie_name, movie_id, idx, freq) in look_up_movie("emma"):
    print(movie_name, movie_id, idx, freq)

emma-2020 485212 4103 454
the-social-dilemma 581924 4541 191


In [8]:
from sklearn.model_selection import train_test_split
from surprise import Dataset, NormalPredictor, Reader


reader = Reader(rating_scale=(1, 10))
# the reader class in surprise just parses ratings on a given scale
# ours are from 1 to 10

dataset = Dataset.load_from_df(data[["user_id", "movie_id", "rating"]], reader)
# load data into surprise dataset class

In [9]:
# from surprise import accuracy, Dataset, SVD
from surprise.model_selection import KFold
from surprise.model_selection import cross_validate
from surprise import (
    accuracy,
    NMF,
    SVD,
    Dataset,
    KNNBasic,
    KNNWithMeans,
    KNNWithZScore,
    Reader,
    SVDpp,
)

# tests the svd algorithm on the dataset as a baseline using 3 fold cross validation

# define a cross-validation iterator
# kf = KFold(n_splits=3)

# algo = SVD()
# algo = KNNBasic()
# algo = KNNWithMeans()
# algo = KNNWithZScore()
# algo = NMF()


# cross_validate(algo, dataset, measures=['RMSE', 'MAE'], cv=5, verbose=True)

# for trainset, testset in kf.split(dataset):

#     # train and test svd algorithm.
#     algo.fit(trainset)
#     predictions = algo.test(testset)

#     # Compute and print Root Mean Squared Error
#     accuracy.rmse(predictions, verbose=True)
#     accuracy.mae(predictions, verbose=True)

In [ ]:
import pandas as pd
import plotly.express as px
from surprise.model_selection import GridSearchCV
from surprise import SVD, NMF, KNNBasic, KNNWithMeans, KNNWithZScore

# Define the experiments
# there're a variety of k's/n_factors including the defaults
configs = {
    'SVD': {
        'class': SVD,
        'params': {'n_factors': [5, 15, 25, 40, 50, 90, 100, 110, 125, 150, 175, 200], 'reg_all': [0.01, 0.1]}
    },
    'NMF': {
        'class': NMF,
        'params': {'n_factors': [5, 10, 15, 25, 50], 'reg_pu': [0.01, 0.06, 0.1]}
    },
    'KNNBasic': {
        'class': KNNBasic,
        'params': {'k': [40, 50, 75, 100], 'sim_options': {'name': ['pearson'], 'user_based': [True]}}
    },
    'KNNWithMeans': {
        'class': KNNWithMeans,
        'params': {'k': [40, 50, 75, 100], 'sim_options': {'name': ['pearson'], 'user_based': [True]}}
    },
    'KNNWithZScore': {
        'class': KNNWithZScore,
        'params': {'k': [40, 50, 75, 100], 'sim_options': {'name': ['pearson'], 'user_based': [True]}}
    }
}

all_results = []

for name, cfg in configs.items():
    print(f"Running grid search for {name}...")
    gs = GridSearchCV(cfg['class'], cfg['params'], measures=['rmse', 'mae'], cv=5)
    gs.fit(dataset)
    
    # Extract all combinations from cv_results
    df_res = pd.DataFrame(gs.cv_results)
    df_res['algorithm'] = name
    # Create a readable string for the parameters
    df_res['params_str'] = df_res['params'].apply(lambda x: str(x))
    
    all_results.append(df_res)

# Combine everything into one master dataframe
full_df = pd.concat(all_results)


Running grid search for SVD...
Running grid search for NMF...
Running grid search for KNNBasic...
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.


In [ ]:
full_df = full_df.sort_values(['mean_test_rmse','mean_test_mae'])

print("RANKED PERFORMANCE Best to Worst per Algorithm")
print()
print(full_df[['algorithm', 'params_str', 'mean_test_rmse', 'std_test_rmse', 'mean_test_mae', 'std_test_mae', 'mean_fit_time', 'mean_test_time']].to_string(index=False))

RANKED PERFORMANCE Best to Worst per Algorithm

    algorithm                                                        params_str  mean_test_rmse  std_test_rmse  mean_fit_time  mean_test_time
          SVD                                 {'n_factors': 50, 'reg_all': 0.1}        1.449561       0.004040       3.083169        1.220099
          SVD                                 {'n_factors': 25, 'reg_all': 0.1}        1.453255       0.003344       2.649706        1.306043
          SVD                                 {'n_factors': 15, 'reg_all': 0.1}        1.455522       0.003487       2.536316        1.317272
          SVD                                 {'n_factors': 10, 'reg_all': 0.1}        1.458969       0.004537       2.349722        1.200250
          SVD                                  {'n_factors': 5, 'reg_all': 0.1}        1.466303       0.001388       2.327418        1.308041
          SVD                                 {'n_factors': 5, 'reg_all': 0.01}        1.466577     